# 1. Install libraries

In [ ]:
!pip install -q numpy pandas scikit-learn h5py

# 2. Clone Github repository

In [ ]:
!rm -rf /content/BTL_ML
!git clone https://github.com/Hoang-Viet-Tran/BTL_ML.git /content/BTL_ML || true
%cd /content/BTL_ML

# 3. Add folder "modules" into Python path

In [ ]:
import sys
sys.path.insert(0, "/content/BTL_ML/modules")

# 4. Download data

In [ ]:
DATA_URL = "https://datasets.imdbws.com/"

!wget -O data.zip $DATA_URL
!unzip -o data.zip -d data

# 5. Import modules

In [ ]:
from data_utils import load_raw_data
from preprocess import preprocess_df
from features import extract_features, save_features

# 6. Load and preprocess data

In [ ]:
df = load_raw_data("/content/BTL_ML/data")
print("Dataset shape:", df.shape)

df = preprocess_df(df)
print("After preprocessing:", df.shape)

df.head()

# 7. Feature extraction

In [ ]:
LABEL_COLUMN = "is_success"

X, y = extract_features(df, label_column=LABEL_COLUMN)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

# 8. Save features

In [ ]:
save_features(X, y, out_dir="/content/BTL_ML/features")

print("✅ Features saved into /features/")
!ls features

# 9. Train model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_val)[:,1]
auc = roc_auc_score(y_val, y_prob)

print("Validation AUC:", auc)

# 10. Save model

In [ ]:
import joblib

joblib.dump(model, "best_model.joblib")
print("✅ Model saved")